In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 1️⃣ Load
df = pd.read_csv('/kaggle/input/e-commerce-review-fixed/shopee_reviews_phase2-labeled-Fixed.csv')

# 2️⃣ Descriptive stats
print("=== DESCRIPTIVE STATS (NUMERIC) ===")
print(df.describe())

# 3️⃣ Descriptive untuk teks (optional)
df['text_length'] = df['content'].fillna('').str.len()
print("\n=== TEXT LENGTH STATS ===")
print(df['text_length'].describe())

# 4️⃣ Missing values
print("\n=== MISSING VALUES ===")
print(df.isna().sum())

# 5️⃣ Duplicate

print("\n=== DUPLICATES ===")
cols = list(df.columns)
subset = cols[cols.index('userName'):cols.index('matched_aspects')+1]
duplicate_count = df.duplicated(subset=subset, keep='first').sum()
print(duplicate_count)

print(f"Columns: {df.columns.tolist()}")

=== DESCRIPTIVE STATS (NUMERIC) ===
             score
count  1043.000000
mean      3.727709
std       1.720562
min       1.000000
25%       2.000000
50%       5.000000
75%       5.000000
max       5.000000

=== TEXT LENGTH STATS ===
count    1043.000000
mean      107.807287
std       105.957504
min         5.000000
25%        39.000000
50%        70.000000
75%       134.000000
max       500.000000
Name: text_length, dtype: float64

=== MISSING VALUES ===
reviewId                    0
userName                    0
userImage                   0
content                     0
score                       0
reviewCreatedVersion      248
at                          0
replyContent               27
repliedAt                  27
matched_aspects             0
Aplikasi                    0
Pengiriman                  0
Produk                      0
Harga                       0
Pembayaran                  0
Layanan Pelanggan (CS)      0
Penjual                     0
text_length                 0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1043 entries, 0 to 1042
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   reviewId                1043 non-null   object
 1   userName                1043 non-null   object
 2   userImage               1043 non-null   object
 3   content                 1043 non-null   object
 4   score                   1043 non-null   int64 
 5   reviewCreatedVersion    795 non-null    object
 6   at                      1043 non-null   object
 7   replyContent            1016 non-null   object
 8   repliedAt               1016 non-null   object
 9   matched_aspects         1043 non-null   object
 10  Aplikasi                1043 non-null   object
 11  Pengiriman              1043 non-null   object
 12  Produk                  1043 non-null   object
 13  Harga                   1043 non-null   object
 14  Pembayaran              1043 non-null   object
 15  Laya

In [3]:
df.head()

,reviewId,userName,userImage,content,score,reviewCreatedVersion,at,replyContent,repliedAt,matched_aspects,Aplikasi,Pengiriman,Produk,Harga,Pembayaran,Layanan Pelanggan (CS),Penjual,text_length
0,31ad33fd-6d00-45d6-b06d-7f69a956c7c0,Aliva Olina,https://play-lh.googleusercontent.com/a/ACg8oc...,"barang2x bagus dan dtng tepat waktu, cuma ongk...",5,3.59.43,2025-10-21 9:38:00,Hi kak Aliva Olina maaf ya terkait keluhan kam...,2025-10-21 10:47:55,"Pengiriman,Produk,Harga",0,P,P,N,0,0,0,70
1,85ed5a52-eabe-4f20-b849-ee7d9e184277,arum galih,https://play-lh.googleusercontent.com/a/ACg8oc...,pengiriman lama poll,1,NaN,2025-10-21 9:37:38,Hai kak arum galih maaf ya bikin ga nyaman ter...,2025-10-21 10:29:48,Pengiriman,0,N,0,0,0,0,0,20
2,b2052463-4a1b-4315-bfa2-4fd139b7136b,Sandi Rama,https://play-lh.googleusercontent.com/a/ACg8oc...,bagus banget aplikasi bermanfaat jika sulit me...,5,3.60.24,2025-10-21 9:36:32,"Hi Kak Sandi Rama, makasih bgt ya review dan b...",2025-10-21 10:07:54,Aplikasi,P,0,0,0,0,0,0,74
3,61843fd3-cfe2-4fa2-8199-b3f27db98e7c,Nar Tiwen,https://play-lh.googleusercontent.com/a/ACg8oc...,padahal nyari lokasi yang sama toko dari indra...,2,NaN,2025-10-21 9:35:22,Hai kak Nar Tiwen\nmaaf yaa udh buat ga nyaman...,2025-10-21 10:11:01,Pengiriman,0,N,0,0,0,0,0,236
4,c760828f-b897-4dfb-aa52-6225d2cfe7c2,Darpin Er,https://play-lh.googleusercontent.com/a/ACg8oc...,saya kasi bintang lima karena cara masuknya mu...,5,NaN,2025-10-21 9:35:01,"Hi Kak Darpin Er , makasih ya buat bintang nya...",2025-10-21 10:10:56,"Aplikasi,Produk",P,0,P,0,0,0,0,257


In [4]:
from IPython.display import display
# Input: 
#  - df: pandas.DataFrame yang memiliki kolom 'content'
#  - min_words: minimal kata agar review dianggap valid (default 2)
#  - keep_single_word_exceptions: list of single-word strings yang TIDAK boleh dihapus (case-insensitive, default ['murah'])
# Proses/cara kerja:
#  - Isi NaN dengan string kosong
#  - Normalisasi ringan (lowercase, strip, hapus tanda baca) untuk menghitung kata
#  - Hitung jumlah kata menggunakan regex \w+ (lebih robust dari split)
#  - Tandai baris dengan jumlah kata < min_words sebagai kandidat penghapusan,
#    kecualikan baris yang setelah normalisasi tepat sama dengan salah satu kata di keep_single_word_exceptions
#  - Tampilkan debugging (daftar baris yang akan dihapus) lalu hapus baris tersebut
# Output:
#  - DataFrame hasil (baris pendek yang tidak dikecualikan telah dihapus)
# Kegunaan:
#  - Membersihkan dataset review dari entri 1-kata/blank, namun mempertahankan kata penting seperti "murah"
def remove_short_reviews_with_exceptions(df, min_words=2, keep_single_word_exceptions=None, content_col='content'):
    if keep_single_word_exceptions is None:
        keep_single_word_exceptions = ['murah']  # default exception words

    # normalisasi daftar pengecualian ke bentuk lowercase tanpa tanda baca/spasi
    def _norm_exception_word(w):
        w = str(w).lower().strip()
        w = re.sub(r"[^\w\s]", "", w)
        return w
    exceptions_norm = set([_norm_exception_word(w) for w in keep_single_word_exceptions if w is not None])

    df = df.copy()
    # isi NaN dengan string kosong supaya aman
    df = df.fillna("")

    # fungsi normalisasi ringan untuk per-barang content (untuk wordcount dan pengecualian)
    def normalize_text(t):
        t = str(t).lower().strip()
        t = re.sub(r"[^\w\s]", "", t)  # hapus tanda baca
        t = re.sub(r"\s+", " ", t)
        return t

    # simpan versi ternormalisasi dan hitung kata (menggunakan \w+)
    df['_normalized_for_wordcount'] = df[content_col].apply(normalize_text)
    df['_word_count'] = df['_normalized_for_wordcount'].apply(lambda x: len(re.findall(r'\w+', x)))

    # kandidat penghapusan: jumlah kata < min_words
    mask_candidate = df['_word_count'] < min_words

    # kecualikan baris yg normalized content tepat sama dengan salah satu exception
    # (contoh: 'murah' akan disimpan meski _word_count == 1)
    mask_exception = df['_normalized_for_wordcount'].apply(lambda x: x in exceptions_norm)

    # final mask yang akan dihapus: kandidat AND NOT exception
    mask_remove = mask_candidate & (~mask_exception)

    to_remove_df = df.loc[mask_remove].copy()   # debugging view
    kept_exceptions_df = df.loc[mask_candidate & mask_exception].copy()  # satu-kata yang dikecualikan

    # Debugging output: tampilkan baris yang akan dihapus dan baris exception yang disimpan
    print(f"=== DEBUG: Baris yg akan DIHAPUS (jumlah kata < {min_words}, kecuali exception) ===")
    if not to_remove_df.empty:
        # tampilkan kolom penting supaya tidak terlalu panjang
        display_cols = [c for c in ['content', '_normalized_for_wordcount', '_word_count', 'score'] if c in to_remove_df.columns]
        print(f"Total akan dihapus: {len(to_remove_df)}")
        display(to_remove_df[display_cols].reset_index(drop=True))
    else:
        print("Tidak ada baris yang memenuhi kriteria penghapusan.")

    if not kept_exceptions_df.empty:
        print(f"\n=== INFO: Baris satu-kata yang DITEPATI pengecualian (disimpan) — exceptions = {sorted(list(exceptions_norm))} ===")
        display_cols = [c for c in ['reviewId','content', '_normalized_for_wordcount', '_word_count', 'score'] if c in kept_exceptions_df.columns]
        display(kept_exceptions_df[display_cols].reset_index(drop=True))

    # ringkasan
    before = len(df)
    removed_count = int(mask_remove.sum())
    after = before - removed_count
    print("\n=== RINGKASAN PENGHAPUSAN ===")
    print(f"Total sebelum: {before}")
    print(f"Baris dihapus: {removed_count}")
    print(f"Total sesudah: {after}")

    # hasil bersih tanpa kolom internal
    df_cleaned = df.loc[~mask_remove].copy().reset_index(drop=True)
    df_cleaned = df_cleaned.drop(columns=['_normalized_for_wordcount', '_word_count'], errors='ignore')

    return df_cleaned
    
df = remove_short_reviews_with_exceptions(df, min_words=2, keep_single_word_exceptions=['murah'])

=== DEBUG: Baris yg akan DIHAPUS (jumlah kata < 2, kecuali exception) ===
Total akan dihapus: 9


,content,_normalized_for_wordcount,_word_count,score
0,mantapp,mantapp,1,5
1,mantapp,mantapp,1,5
2,respon,respon,1,5
3,mntapppp,mntapppp,1,5
4,mantappp,mantappp,1,5
5,Mantaapppppppp....,mantaapppppppp,1,5
6,mantaaaappp,mantaaaappp,1,5
7,mantappp,mantappp,1,5
8,mantapp,mantapp,1,5



=== INFO: Baris satu-kata yang DITEPATI pengecualian (disimpan) — exceptions = ['murah'] ===


,reviewId,content,_normalized_for_wordcount,_word_count,score
0,c7d3acca-b219-44b3-b76d-8f8ea451b489,murah,murah,1,5
1,d7607e6b-1007-4eec-aa32-1e3e6af7c451,murah,murah,1,5



=== RINGKASAN PENGHAPUSAN ===
Total sebelum: 1043
Baris dihapus: 9
Total sesudah: 1034


In [5]:
# -----------------------------------------------------------------------------
# Input:
#   df: pandas.DataFrame yang memuat kolom 'matched_aspects' dan kolom aspek (mis. 'Aplikasi', 'Pengiriman', ...)
#   aspects: list nama kolom aspek yang akan dicek (default daftar yang Anda sebutkan)
#   matched_col: nama kolom yang berisi aspek yang cocok (string, default 'matched_aspects')
#   label_values: set nilai label yang dianggap (default {'P','N'})
#   save_path: jika tidak None, simpan hasil DataFrame bermasalah ke path CSV ini
#
# Proses / cara kerja:
#   1. Isi NaN di df dengan string kosong untuk menghindari error.
#   2. Normalisasi isi kolom matched_aspects -> split by comma, strip, lower-case.
#   3. Untuk setiap row dan setiap aspek di `aspects`:
#        - ambil nilai label kolom aspek, normalisasikan (strip, upper)
#        - jika label termasuk label_values (P/N) tetapi nama aspek tidak ada di matched_aspects (case-insensitive),
#          tandai aspek tersebut sebagai "missing_in_matched"
#   4. Buat kolom bantu: 'missing_aspects_list' (list nama aspek yang bermasalah) dan
#      'missing_aspects_details' (e.g. "Penjual(P); Aplikasi(N)").
#   5. Filter hanya baris yang memiliki missing_aspects_list non-empty -> ini hasil yang dikembalikan.
#
# Output:
#   - problems_df: DataFrame baris yang bermasalah, dengan kolom-kolom asli + 'missing_aspects_list' + 'missing_aspects_details'
#   - summary: dict ringkasan berisi jumlah total masalah dan per-aspect counts
#
# Kegunaan:
#   - Memeriksa konsistensi antara label aspek (P/N) dan hasil ekstraksi `matched_aspects`.
#   - Memudahkan reviewer untuk memperbaiki anotasi yang terlewat.
# -----------------------------------------------------------------------------
def find_aspect_label_mismatches(df,
                                 aspects=None,
                                 matched_col='matched_aspects',
                                 label_values=None,
                                 save_path=None):
    if aspects is None:
        aspects = ['Aplikasi', 'Pengiriman', 'Produk', 'Harga', 'Pembayaran',
                   'Layanan Pelanggan (CS)', 'Penjual']
    if label_values is None:
        label_values = {'P', 'N'}

    # Salin df dan isi NA
    df = df.copy().fillna('')

    # Normalisasi matched_aspects -> menghasilkan set lowercased names per row
    def parse_matched(s):
        if s is None:
            return set()
        # split on comma, strip spaces, lower-case; ignore empty tokens
        tokens = [t.strip().lower() for t in str(s).split(',') if t.strip() != '']
        return set(tokens)

    matched_sets = df[matched_col].apply(parse_matched)

    # Persiapkan hasil
    missing_list = []            # list of lists (nama aspek yang missing)
    missing_details = []         # string details like "Penjual(P); Aplikasi(N)"
    per_aspect_count = {a: 0 for a in aspects}
    total_rows_with_issues = 0

    for idx, row in df.iterrows():
        matched = matched_sets.loc[idx]  # set of normalized matched aspects
        row_missing = []
        row_missing_details = []

        for aspect in aspects:
            # ambil label value di kolom aspek, normalisasi
            label_raw = str(row.get(aspect, '')).strip()
            label = label_raw.upper()  # P, N, atau lainnya

            # normalize aspect name for comparison: lower-case and strip
            aspect_norm = str(aspect).strip().lower()

            # jika label termasuk P/N dan aspect_norm tidak ada di matched -> catat
            if label in label_values and aspect_norm not in matched:
                row_missing.append(aspect)
                row_missing_details.append(f"{aspect}({label})")
                per_aspect_count[aspect] += 1

        if row_missing:
            total_rows_with_issues += 1
        missing_list.append(row_missing)
        missing_details.append('; '.join(row_missing_details))

    # Masukkan ke df hasil
    result_df = df.copy()
    result_df['missing_aspects_list'] = missing_list
    result_df['missing_aspects_details'] = missing_details

    # Filter hanya baris yang bermasalah
    problems_df = result_df[result_df['missing_aspects_list'].apply(lambda x: len(x) > 0)].copy().reset_index(drop=True)

    # Summary
    summary = {
        'total_rows_checked': int(len(df)),
        'total_rows_with_issues': int(total_rows_with_issues),
        'per_aspect_count': per_aspect_count
    }

    # Opsional simpan
    if save_path:
        problems_df.to_csv(save_path, index=False)

    return problems_df, summary


In [6]:
problems_df, summary = find_aspect_label_mismatches(df,
                                                    aspects=['Aplikasi','Pengiriman','Produk','Harga','Pembayaran',
                                                             'Layanan Pelanggan (CS)','Penjual'],
                                                    matched_col='matched_aspects',
                                                    label_values={'P','N'},
                                                    save_path='/kaggle/working/aspect_label_mismatches.csv'  # optional
                                                   )

# 3) Ringkasan & preview
print("=== SUMMARY ===")
print(f"Total rows checked: {summary['total_rows_checked']}")
print(f"Total rows with mismatches: {summary['total_rows_with_issues']}")
print("Per-aspect missing counts:")
for a, c in summary['per_aspect_count'].items():
    print(f"  {a}: {c}")

print("\n=== baris yang bermasalah ===")
if len(problems_df) > 0:
    # tampilkan kolom penting + detail supaya mudah inspeksi
    display_cols = [c for c in ['reviewId','content','matched_aspects','missing_aspects_list','missing_aspects_details'] if c in problems_df.columns]
    display(problems_df[display_cols])
else:
    print("Tidak ditemukan baris dengan mismatch.")

=== SUMMARY ===
Total rows checked: 1034
Total rows with mismatches: 0
Per-aspect missing counts:
  Aplikasi: 0
  Pengiriman: 0
  Produk: 0
  Harga: 0
  Pembayaran: 0
  Layanan Pelanggan (CS): 0
  Penjual: 0

=== baris yang bermasalah ===
Tidak ditemukan baris dengan mismatch.


In [7]:
# Input:
#  - df: pandas.DataFrame yang memuat kolom 'matched_aspects' dan kolom aspek (mis. 'Aplikasi', 'Pengiriman', ...)
#  - aspects: list nama kolom aspek yang akan dicek (default sesuai permintaan)
#  - matched_col: nama kolom yang memuat hasil ekstraksi aspek (default 'matched_aspects')
#  - valid_labels: set label yang dianggap valid untuk aspek (default {'P','N'})
#  - save_path: jika tidak None, simpan hasil ke CSV pada path ini
#
# Proses / cara kerja:
#  1. Isi NaN menjadi string kosong.
#  2. Normalisasi token dalam matched_aspects (split by comma, strip, lowercase, hapus tanda baca minimal).
#  3. Buat mapping antara bentuk normal aspek (lower tanpa tanda baca/spasi ekstra) ke nama kolom aslinya.
#  4. Untuk setiap row: periksa setiap token di matched_aspects, temukan kolom aspek sesuai mapping,
#     ambil nilai label di kolom tersebut, normalisasi (strip, upper) dan cek apakah termasuk valid_labels.
#  5. Jika tidak valid, catat row tersebut beserta detail (nama aspek, nilai label yang ditemukan).
#
# Output:
#  - problems_df: DataFrame berisi baris-baris yang bermasalah + kolom 'problem_aspect' & 'found_label'
#  - summary: dict ringkasan (total rows checked, total problem rows, per-aspect counts)
#
# Kegunaan:
#  - Menemukan inkonsistensi: aspek tercantum di matched_aspects tetapi kolomnya belum dilabeli P/N.
def find_matched_but_unlabeled(df,
                               aspects=None,
                               matched_col='matched_aspects',
                               valid_labels=None,
                               save_path=None):
    if aspects is None:
        aspects = ['Aplikasi', 'Pengiriman', 'Produk', 'Harga', 'Pembayaran',
                   'Layanan Pelanggan (CS)', 'Penjual']
    if valid_labels is None:
        valid_labels = {'P', 'N'}

    df = df.copy().fillna('')

    # helper: normalisasi nama aspek/token untuk perbandingan
    def norm_token(s):
        s = str(s).lower().strip()
        # hapus tanda baca seperti () , . - dan multiple spasi
        s = re.sub(r'[^\w\s]', '', s)
        s = re.sub(r'\s+', ' ', s)
        return s

    # siapkan mapping: normalized aspect name -> original column name
    aspect_norm_to_col = {}
    for a in aspects:
        aspect_norm_to_col[norm_token(a)] = a

    # parse matched_aspects jadi list token normal
    def parse_matched(s):
        if s is None:
            return []
        # split by comma, strip, ignore empty tokens
        tokens = [t.strip() for t in str(s).split(',') if t.strip() != '']
        # normalisasi tiap token
        return [norm_token(t) for t in tokens]

    problem_rows = []    # akan berisi dict per masalah (bisa ada multiple per row)
    per_aspect_count = {a: 0 for a in aspects}
    rows_checked = 0

    for idx, row in df.iterrows():
        rows_checked += 1
        matched_tokens = parse_matched(row.get(matched_col, ''))
        if not matched_tokens:
            continue

        # untuk tiap token di matched_aspects, cari kolomnya lalu periksa label
        for tok in matched_tokens:
            colname = aspect_norm_to_col.get(tok)  # None jika token tidak match salah satu aspek yang kita awasi
            if colname is None:
                # token matched_aspects bukan salah satu aspek yang dipantau -> lewati
                continue

            label_raw = str(row.get(colname, '')).strip()
            label_norm = label_raw.upper()

            # jika label tidak termasuk valid_labels => masalah (tercantum tapi tidak dilabeli P/N)
            if label_norm not in valid_labels:
                per_aspect_count[colname] += 1
                problem_rows.append({
                    'orig_index': idx,
                    'reviewId': row.get('reviewId', ''),
                    'content': row.get('content', ''),
                    'matched_aspects': row.get(matched_col, ''),
                    'problem_aspect': colname,
                    'found_label': label_raw,
                    'row_data': row.to_dict()   # opsional: bisa dipakai untuk debugging lebih lanjut
                })

    # buat DataFrame hasil (jika tidak ada masalah, kosong)
    if problem_rows:
        problems_df = pd.DataFrame(problem_rows)
    else:
        problems_df = pd.DataFrame(columns=['orig_index','reviewId','content','matched_aspects','problem_aspect','found_label','row_data'])

    summary = {
        'total_rows_checked': int(rows_checked),
        'total_problem_rows': int(problems_df['orig_index'].nunique()) if not problems_df.empty else 0,
        'total_problem_entries': int(len(problems_df)),
        'per_aspect_count': per_aspect_count
    }

    # simpan jika diminta
    if save_path and not problems_df.empty:
        # simpan tanpa kolom row_data (bisa jadi sangat besar / tidak serializable ke CSV dengan baik)
        to_save = problems_df.drop(columns=['row_data'], errors='ignore')
        to_save.to_csv(save_path, index=False)

    return problems_df, summary

In [8]:
problems_df, summary = find_matched_but_unlabeled(df,
                                                  aspects=['Aplikasi','Pengiriman','Produk','Harga','Pembayaran',
                                                           'Layanan Pelanggan (CS)','Penjual'],
                                                  matched_col='matched_aspects',
                                                  valid_labels={'P','N'},
                                                  save_path='/kaggle/working/matched_but_unlabeled.csv'  # optional
                                                 )

# ringkasan
print("=== SUMMARY ===")
print(f"Total rows checked: {summary['total_rows_checked']}")
print(f"Total distinct rows with at least one issue: {summary['total_problem_rows']}")
print(f"Total problem entries (row x aspect): {summary['total_problem_entries']}")
print("Per-aspect problem counts:")
for a, c in summary['per_aspect_count'].items():
    print(f"  {a}: {c}")

# preview: tunjukkan beberapa entri masalah
if not problems_df.empty:
    display_cols = ['orig_index','reviewId','content','matched_aspects','problem_aspect','found_label']
    display(problems_df[display_cols].head(200))
else:
    print("Tidak ditemukan kasus matched_aspects tanpa label P/N pada kolom aspek yang diawasi.")

=== SUMMARY ===
Total rows checked: 1034
Total distinct rows with at least one issue: 0
Total problem entries (row x aspect): 0
Per-aspect problem counts:
  Aplikasi: 0
  Pengiriman: 0
  Produk: 0
  Harga: 0
  Pembayaran: 0
  Layanan Pelanggan (CS): 0
  Penjual: 0
Tidak ditemukan kasus matched_aspects tanpa label P/N pada kolom aspek yang diawasi.


In [9]:
!pip install Sastrawi pyahocorasick emoji unidecode requests --quiet

import json
import time
import pickle
import tempfile
import hashlib
from io import StringIO
from datetime import datetime, timezone

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import emoji
from unidecode import unidecode

# Sastrawi untuk stemming dan stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print("✅ All libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.6 MB/s eta 0:00:00
✅ All libraries imported successfully!


In [10]:
# Project directories
PROJECT_DIR = "/kaggle/working"
CACHE_DIR = os.path.join(PROJECT_DIR, "automaton_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Cache file paths
LOCAL_SLANG_CSV = os.path.join(CACHE_DIR, "slang_dict_loaded.csv")
LOCAL_SLANG_PKL = os.path.join(CACHE_DIR, "slang_dict_loaded.pkl")
AUTOMATON_PKL = os.path.join(CACHE_DIR, "automaton.pkl")
METADATA_JSON = os.path.join(CACHE_DIR, "slang_metadata.json")

# Slang dictionary sources (same as Phase 1)
SLANG_URLS = [
    "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv",
    "https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/new_kamusalay.csv"
]

print(f"✅ Configuration loaded!")
print(f"   Project dir: {PROJECT_DIR}")
print(f"   Cache dir: {CACHE_DIR}")
print("\n🔧 Initializing sentiment-aware stopwords...")

# Load stopwords dari Sastrawi
factory = StopWordRemoverFactory()
INDO_STOPWORDS_BASE = set(factory.get_stop_words())

# Custom stopwords tambahan (non-sentiment-critical)
CUSTOM_STOPWORDS = {
    'nya', 'lah', 'kah', 'ini', 'itu', 'sih', 'kok', 'dong', 'deh',
    'yg', 'utk', 'dgn', 'krn', 'akan', 'telah', 'banget', 'paling',
    'sangat', 'amat'
}

# Kata-kata yang HARUS DIPERTAHANKAN untuk sentiment analysis
SENTIMENT_CRITICAL_WORDS = {
    # Negation words - CRITICAL untuk sentiment flip
    "tidak", "bukan", "belum", "jangan", "tanpa"
    
    # Adversative conjunctions - menunjukkan kontras/trade-off
    'tapi', 'tetapi', 'namun', 'walau', 'walaupun', 'meskipun', 'sedangkan', 'melainkan',
    
    # Exceptive words - menunjukkan exception
    'kecuali', 'selain',
    
    # Diminishers - bisa affect sentiment strength menuju threshold
    'kurang', 'agak', 'cukup', 'lumayan',
    
    # Temporal yang bisa affect sentiment
    'masih', 'sudah',
}

# Combine stopwords
ALL_STOPWORDS = INDO_STOPWORDS_BASE | CUSTOM_STOPWORDS

# Remove sentiment-critical words dari stopwords
# Ini memastikan kata-kata penting tidak ter-remove
INDO_STOPWORDS_FINAL = ALL_STOPWORDS - SENTIMENT_CRITICAL_WORDS

# Create preserve tokens set (normalized)
PRESERVE_TOKENS = set()
for word in SENTIMENT_CRITICAL_WORDS:
    word_norm = re.sub(r"[^\w\s]", "", word.lower()).strip()
    if word_norm:
        PRESERVE_TOKENS.add(word_norm)

print(f"✅ Stopwords configuration:")
print(f"   Base stopwords (Sastrawi): {len(INDO_STOPWORDS_BASE)}")
print(f"   Custom stopwords added: {len(CUSTOM_STOPWORDS)}")
print(f"   Sentiment-critical words (preserved): {len(SENTIMENT_CRITICAL_WORDS)}")
print(f"   Final stopwords (after filtering): {len(INDO_STOPWORDS_FINAL)}")
print(f"\n🔒 Preserved words: {sorted(PRESERVE_TOKENS)}")

print("\n🔧 Initializing stemmer...")
stemmer = StemmerFactory().create_stemmer()
print("✅ Stemmer ready!")

✅ Configuration loaded!
   Project dir: /kaggle/working
   Cache dir: /kaggle/working/automaton_cache

🔧 Initializing sentiment-aware stopwords...
✅ Stopwords configuration:
   Base stopwords (Sastrawi): 123
   Custom stopwords added: 19
   Sentiment-critical words (preserved): 20
   Final stopwords (after filtering): 124

🔒 Preserved words: ['agak', 'belum', 'bukan', 'cukup', 'jangan', 'kecuali', 'kurang', 'lumayan', 'masih', 'melainkan', 'meskipun', 'namun', 'sedangkan', 'selain', 'sudah', 'tanpatapi', 'tetapi', 'tidak', 'walau', 'walaupun']

🔧 Initializing stemmer...
✅ Stemmer ready!


In [11]:
# Input: text (str) — teks yang ingin di-hash SHA1
# Proses:
#   1. Meng-encode teks ke bytes menggunakan UTF-8.
#   2. Memanggil hashlib.sha1() untuk menghitung hash SHA1 dari bytes tersebut.
#   3. Mengubah hasil hash menjadi representasi heksadesimal string lewat .hexdigest().
# Output: (str) — string hex 40-karakter yang merupakan nilai SHA1 dari input.
# Kegunaan:
#   - Membuat fingerprint/checksum singkat untuk teks (mis. content-ID, cache key, verifikasi integritas).
#   - Tidak cocok untuk keamanan kriptografi modern (gunakan SHA256/algoritma lebih kuat jika butuh keamanan).
def atomic_write_text(path, text, enc='utf-8'):
    dirn = os.path.dirname(path)
    os.makedirs(dirn, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=dirn)
    try:
        with os.fdopen(fd, 'w', encoding=enc) as f:
            f.write(text)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        except:
            pass
        raise

def atomic_write_pickle(path, obj):
    dirn = os.path.dirname(path)
    os.makedirs(dirn, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=dirn)
    try:
        with os.fdopen(fd, 'wb') as f:
            pickle.dump(obj, f)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        except:
            pass
        raise
# Input: text (str) — teks yang ingin di-hash SHA1
# Proses:
#   1. Meng-encode teks ke bytes menggunakan UTF-8.
#   2. Memanggil hashlib.sha1() untuk menghitung hash SHA1 dari bytes tersebut.
#   3. Mengubah hasil hash menjadi representasi heksadesimal string lewat .hexdigest().
# Output: (str) — string hex 40-karakter yang merupakan nilai SHA1 dari input.
# Kegunaan:
#   - Membuat fingerprint/checksum singkat untuk teks (mis. content-ID, cache key, verifikasi integritas).
#   - Tidak cocok untuk keamanan kriptografi modern (gunakan SHA256/algoritma lebih kuat jika butuh keamanan).
def compute_sha1(text):
    return hashlib.sha1(text.encode('utf-8')).hexdigest()

print("✅ Utility functions loaded!")

✅ Utility functions loaded!


In [12]:
# Input:
#   - retries (int): jumlah percobaan ulang maksimum untuk request (default: 3)
#   - backoff_factor (float): faktor pengali untuk jeda antar retry (sesuai mekanisme exponential backoff)
# Proses:
#   1. Buat objek requests.Session() untuk koneksi HTTP yang dipertahankan.
#   2. Konfigurasi objek Retry dari urllib3 (melalui requests.adapters.Retry) dengan parameter:
#        - total, read, connect = retries
#        - backoff_factor = backoff_factor (mengatur jeda bertambah tiap retry)
#        - status_forcelist = (500, 502, 504) (kode status yang akan memicu retry)
#   3. Buat HTTPAdapter dengan max_retries=retry dan mount adapter tersebut ke session untuk 'http://' dan 'https://'.
#   4. Kembalikan session yang sudah terpasang mekanisme retry.
# Output:
#   - requests.Session() siap pakai yang otomatis melakukan retry sesuai konfigurasi pada kegagalan koneksi/read atau status tertentu.
# Kegunaan:
#   - Membuat panggilan HTTP lebih tangguh terhadap gangguan sementara (timeout, 502, 500, dll.)
#   - Cocok digunakan untuk memanggil API eksternal (mis. reviews(...) jika memakai requests)
# Catatan:
#   - Konfigurasi Retry dan status_forcelist bisa disesuaikan dengan kebutuhan API target.
#   - Pastikan juga mengatur timeout pada request (mis. session.get(..., timeout=5)) untuk mencegah blocking lama.
#   - Pada beberapa versi requests/urllib3, import path untuk Retry bisa berbeda: from urllib3.util.retry import Retry
def make_session_with_retry(retries=3, backoff_factor=0.3):
    session = requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=(500, 502, 504),
        allowed_methods=frozenset(['GET', 'POST'])
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

print("✅ HTTP session factory ready!")

✅ HTTP session factory ready!


In [13]:
# Input:
#   - url (str): URL yang akan di-fetch (teks/CSV)
#   - session (requests.Session): session HTTP (disarankan dibuat via make_session_with_retry)
#   - prev_info (dict|None): metadata sebelumnya untuk sumber ini (mengandung 'etag' atau 'last_modified'), default None
#   - timeout (int|float): timeout (detik) untuk request GET, default 15
# Proses / Cara kerja (detail):
#   1. Siapkan header kosong. Jika prev_info diberikan dan mengandung 'etag' atau 'last_modified',
#      tambahkan header kondisional 'If-None-Match' atau 'If-Modified-Since' agar server mengembalikan 304
#      bila sumber tidak berubah.
#   2. Lakukan HTTP GET menggunakan session.get(url, headers=headers, timeout=timeout).
#   3. Jika server mengembalikan status_code 304 -> kembalikan ('not_modified', None, info) agar caller tahu
#      bahwa sumber belum berubah (bisa gunakan cache lokal).
#   4. Jika status_code 200 -> baca r.text, hitung SHA1 teks, ambil header ETag dan Last-Modified bila ada,
#      bangun dict info berisi status_code, etag, last_modified, sha1, url, lalu kembalikan ('fetched', text, info).
#   5. Untuk status selain 200/304 -> kembalikan ('error', None, {'status_code': r.status_code, 'url': url}).
#   6. Tangani exception jaringan/timeout dengan menangkap Exception dan mengembalikan ('error', None, {'error': str(e), 'url': url}).
# Output:
#   - Tuple (status, text_or_none, info_dict)
#       * status: 'fetched' | 'not_modified' | 'error'
#       * text_or_none: teks konten jika fetched, None otherwise
#       * info_dict: metadata seperti status_code, etag, last_modified, sha1, url atau error message
# Kegunaan:
#   - Mengambil konten sumber eksternal secara efisien dengan dukungan caching berbasis ETag/Last-Modified.
#   - Digunakan oleh pipeline yang ingin memperbarui data hanya bila sumber berubah, menghemat bandwidth dan waktu.
# Catatan:
#   - Mengandalkan session untuk konfigurasi timeout/retry yang tepat. Pastikan juga menangani encoding/size jika sumber besar.
def conditional_fetch_text(url, session, prev_info=None, timeout=15):
    headers = {}
    if prev_info:
        if prev_info.get('etag'):
            headers['If-None-Match'] = prev_info.get('etag')
        elif prev_info.get('last_modified'):
            headers['If-Modified-Since'] = prev_info.get('last_modified')
    
    try:
        r = session.get(url, headers=headers, timeout=timeout)
        
        if r.status_code == 304:
            return 'not_modified', None, {'status_code': 304, 'url': url}
        
        if r.status_code == 200:
            text = r.text
            info = {
                'status_code': 200,
                'etag': r.headers.get('ETag'),
                'last_modified': r.headers.get('Last-Modified'),
                'sha1': compute_sha1(text),
                'url': url
            }
            return 'fetched', text, info
        
        return 'error', None, {'status_code': r.status_code, 'url': url}
    
    except Exception as e:
        return 'error', None, {'error': str(e), 'url': url}
# Input:
#   - urls (list[str]): daftar URL CSV yang berisi pasangan slang -> formal
#   - cache_csv (str): path file CSV lokal untuk menyimpan hasil gabungan (default LOCAL_SLANG_CSV)
#   - cache_pkl (str): path file pickle untuk cache ter-serialisasi (default LOCAL_SLANG_PKL)
#   - metadata_json (str): path file JSON metadata (default METADATA_JSON)
#   - session (requests.Session|None): session HTTP; jika None akan dibuat dengan make_session_with_retry()
#   - max_age_days (int): umur cache maksimal (hari) sebelum dianggap kadaluarsa, default 30
# Proses / Cara kerja (detail):
#   1. Jika session None -> buat session via make_session_with_retry().
#   2. Muat metadata dari metadata_json bila ada; jika gagal baca, lanjut dengan meta kosong.
#   3. Hitung umur cache (age_days) dari meta['fetched_at'] bila tersedia, dengan konversi ISO -> datetime.
#   4. Jika cache_pkl ada dan age_days tersedia dan < max_age_days:
#        - Coba load pickle, print penggunaan cache, lalu return mapping + meta (menghindari fetch).
#   5. Jika tidak menggunakan cache langsung: coba muat combined_mapping awal dari cache_pkl bila file ada.
#   6. Siapkan prev_sources dari meta['sources'] untuk memberi prev_info ke conditional_fetch_text per URL.
#   7. Iterasi setiap URL:
#        a. Panggil conditional_fetch_text(url, session, prev_info=prev_info).
#        b. Jika status 'not_modified' -> tambahkan prev_info ke new_sources_meta (pakai cache lokal).
#        c. Jika 'fetched' -> parsing CSV dari text menggunakan pandas.read_csv(StringIO(text)).
#           - Tentukan kolom teks (obj dtype). Bila ada >=2 kolom teks, gunakan kolom pertama sebagai key,
#             kolom kedua sebagai value; jika tidak, fallback ke iloc[:,0] dan iloc[:,1].
#           - Normalisasi (strip, lower) lalu buat dict mapping baru dan update combined_mapping.
#           - Tambahkan info ke new_sources_meta, tandai updated = True.
#        d. Jika error -> catat error di new_sources_meta.
#   8. Setelah iterasi, jika ada update atau cache_pkl belum ada:
#        - Simpan combined_mapping ke CSV (cache_csv) dan pickle (cache_pkl) secara atomik.
#        - Bangun meta_new berisi sources, fetched_at (UTC ISO), num_entries, dan sha1 dari JSON terurut mapping.
#        - Simpan metadata_json menggunakan atomic_write_text.
#        - Print notifikasi dan return combined_mapping, meta_new.
#   9. Jika gagal persist, laporkan warning tapi tetap return combined_mapping, meta (meta lama).
# Output:
#   - Tuple (combined_mapping, meta) di mana combined_mapping adalah dict slang->formal gabungan,
#     dan meta adalah metadata terbaru atau meta lama jika tidak ada perubahan.
# Kegunaan:
#   - Mengumpulkan dan menyatukan beberapa sumber kamus slang menjadi satu mapping yang dapat digunakan untuk
#     normalisasi teks (mis. preprocessing review).
#   - Menyediakan caching dan metadata (fetched_at, sha1) sehingga fetch ulang hanya terjadi bila sumber berubah
#     atau cache sudah kadaluarsa.
# Catatan / Perhatian:
#   - Parsing CSV mengasumsikan setidaknya dua kolom; lakukan validasi tambahan bila struktur CSV bervariasi.
#   - Pickle tidak aman untuk data dari sumber tak tepercaya; namun di sini hanya untuk cache lokal internal.
#   - Pastikan konstanta LOCAL_SLANG_CSV, LOCAL_SLANG_PKL, METADATA_JSON didefinisikan; dan impor yang diperlukan:
#     os, json, pickle, pandas as pd, StringIO, datetime, timezone, compute_sha1, atomic_write_pickle, atomic_write_text.
#   - Untuk sumber besar, pertimbangkan streaming parsing daripada memuat seluruh teks ke memori.
def load_mapping_with_metadata(urls, cache_csv=LOCAL_SLANG_CSV, 
                               cache_pkl=LOCAL_SLANG_PKL,
                               metadata_json=METADATA_JSON, 
                               session=None, max_age_days=30):
    if session is None:
        session = make_session_with_retry()
    
    # Load existing metadata
    meta = {}
    if os.path.exists(metadata_json):
        try:
            with open(metadata_json, 'r', encoding='utf-8') as f:
                meta = json.load(f)
        except Exception:
            meta = {}
    
    # Calculate cache age
    age_days = None
    if meta.get('fetched_at'):
        try:
            iso = meta['fetched_at']
            if iso.endswith('Z'):
                iso = iso.replace('Z', '+00:00')
            fetched_dt = datetime.fromisoformat(iso)
            age_days = (datetime.now(timezone.utc) - fetched_dt).days
        except Exception:
            age_days = None
    
    # Try to use cache if fresh
    if os.path.exists(cache_pkl) and age_days is not None and age_days < max_age_days:
        try:
            with open(cache_pkl, 'rb') as f:
                mapping = pickle.load(f)
            print(f"📂 Using cached slang dictionary (age {age_days} days, {len(mapping)} entries)")
            return mapping, meta
        except Exception:
            print("⚠️ Cache load failed, will fetch fresh data")
    
    # Fetch fresh data
    combined = {}
    prev_sources = {s.get('url'): s for s in meta.get('sources', [])} if meta.get('sources') else {}
    new_sources_meta = []
    updated = False

    if os.path.exists(cache_pkl):
        try:
            with open(cache_pkl, 'rb') as f:
                combined_mapping = pickle.load(f) or {}
        except Exception:
            combined_mapping = {}
    
    print("📥 Fetching slang dictionaries from online sources...")
    
    for url in urls:
        prev_info = prev_sources.get(url, {})
        status, text, info = conditional_fetch_text(url, session, prev_info=prev_info)
        
        if status == 'not_modified':
            print(f"  ℹ️  {url.split('/')[-1]}: not modified (using previous)")
            new_sources_meta.append(prev_info)
            continue
        
        if status == 'fetched' and text:
            try:
                df_src = pd.read_csv(StringIO(text))
                
                # Identify text columns (assuming first two are slang -> formal)
                txt_cols = [c for c in df_src.columns if df_src[c].dtype == object]
                
                if len(txt_cols) >= 2:
                    kcol, vcol = txt_cols[0], txt_cols[1]
                    new_map = dict(zip(
                        df_src[kcol].astype(str).str.strip().str.lower(),
                        df_src[vcol].astype(str).str.strip().str.lower()
                    ))
                else:
                    # Fallback: use first two columns
                    new_map = dict(zip(
                        df_src.iloc[:, 0].astype(str).str.strip().str.lower(),
                        df_src.iloc[:, 1].astype(str).str.strip().str.lower()
                    ))
                
                combined.update(new_map)
                new_sources_meta.append(info)
                updated = True
                print(f"  ✅ {url.split('/')[-1]}: fetched {len(new_map)} entries")
                
            except Exception as e:
                print(f"  ❌ Failed parsing {url}: {e}")
                new_sources_meta.append({'url': url, 'error': str(e)})
        else:
            print(f"  ❌ Failed fetching {url}")
            new_sources_meta.append(info)
    
    # Persist cache if updated
    if updated or not os.path.exists(cache_pkl):
        try:
            # Save CSV version for human inspection
            pd.DataFrame(list(combined.items()), columns=['slang', 'formal']).to_csv(
                cache_csv, index=False, encoding='utf-8-sig'
            )
            
            # Save pickle for fast loading
            atomic_write_pickle(cache_pkl, combined)
            
            # Update metadata
            meta_new = {
                'sources': new_sources_meta,
                'fetched_at': datetime.now(timezone.utc).isoformat(),
                'num_entries': len(combined),
                'sha1': compute_sha1(json.dumps(sorted(list(combined.items())), ensure_ascii=False))
            }
            atomic_write_text(metadata_json, json.dumps(meta_new, indent=2, ensure_ascii=False))
            
            print(f"💾 Saved slang dictionary: {len(combined)} entries")
            return combined, meta_new
            
        except Exception as e:
            print(f"⚠️ Failed to persist slang cache: {e}")
    
    return combined, meta

print("✅ Slang dictionary loading functions ready!")

✅ Slang dictionary loading functions ready!


In [14]:
import ahocorasick
# Input:
#   - mapping (dict): dictionary mapping pattern (str) -> replacement (str). Boleh kosong atau None.
# Proses / Cara kerja (detail):
#   1. Jika mapping falsy (None/empty) langsung kembalikan None.
#   2. Normalisasi mapping ke clean_map:
#        - ubah key dan value ke str, strip whitespace, ubah ke lowercase
#        - abaikan key yang setelah strip kosong
#   3. Buat instance ahocorasick.Automaton().
#   4. Untuk setiap pasangan (key, val) pada clean_map, tambahkan pola ke automaton menggunakan A.add_word(key, (key, val)).
#      Menyimpan tuple (key,val) sebagai payload agar saat match kita bisa tahu pengganti/metadatanya.
#   5. Panggil A.make_automaton() untuk membangun struktur internal (fail links) — ukur waktu pembuatan untuk logging.
#   6. Cetak info jumlah pola dan waktu pembuatan.
# Output:
#   - Mengembalikan objek ahocorasick.Automaton yang sudah di-build, atau None jika input kosong.
# Kegunaan:
#   - Menyediakan mesin pencocokan multi-pattern yang sangat cepat untuk mencari banyak kata/frasa sekaligus.
#   - Cocok untuk tugas seperti normalisasi slang, deteksi frasa, token replacement, atau ekstraksi entitas berbasis kamus.
# Catatan / Perhatian:
#   - Memerlukan pustaka `ahocorasick` (pyahocorasick). Pastikan terinstal.
#   - Untuk mapping sangat besar, pembuatan automaton bisa membutuhkan memori dan waktu non-trivial.
#   - Automaton ini melakukan pencocokan substring (bukan token-boundary-aware). Jika butuh boundary-aware, lakukan preprocessing.
def build_aho_automaton(mapping):
    if not mapping:
        return None
    
    # Clean mapping: ensure all keys/values are lowercase strings
    clean_map = {
        str(k).strip().lower(): str(v).strip().lower() 
        for k, v in mapping.items() 
        if str(k).strip() != ""
    }
    
    # Build automaton
    A = ahocorasick.Automaton()
    for k, v in clean_map.items():
        A.add_word(k, (k, v))
    
    start = time.time()
    A.make_automaton()
    elapsed = time.time() - start
    
    print(f"⚙️  Built Aho-Corasick automaton: {len(clean_map)} patterns in {elapsed:.3f}s")
    return A
# Input:
#   - mapping (dict): mapping pattern->replacement yang menjadi sumber pembuatan automaton.
#   - automaton_pkl (str): path file pickle untuk menyimpan/ memuat automaton hasil build.
#   - metadata_json (str): path file JSON metadata yang diasumsikan mengandung 'sha1' dan mungkin info lain.
# Proses / Cara kerja (detail):
#   1. Hitung current_sha: SHA1 dari JSON terurut dari mapping (untuk deteksi perubahan).
#   2. Muat metadata dari metadata_json bila file ada (aman dengan try/except).
#   3. Ambil meta_sha = meta.get('sha1').
#   4. Jika file automaton_pkl ada dan meta_sha sama dengan current_sha:
#        - Coba buka automaton_pkl (pickle.load) dan kembalikan automaton yang di-cache.
#        - Jika gagal (exception), tampilkan peringatan dan lanjut ke rebuild.
#   5. Jika tidak ada cache valid, panggil build_aho_automaton(mapping) untuk membuat automaton baru.
#   6. Coba simpan automaton yang dibangun ke automaton_pkl menggunakan atomic_write_pickle untuk menulis atomik.
#      - Update meta (mis. meta['automaton_saved_at']) dan simpan kembali metadata_json secara atomik.
#      - Jika penyimpanan gagal, tampilkan peringatan tapi tetap kembalikan automaton yang dibangun.
# Output:
#   - Mengembalikan objek ahocorasick.Automaton (baik yang dimuat dari cache maupun yang baru dibangun).
# Kegunaan:
#   - Menghindari rebuild automaton yang mahal jika mapping tidak berubah (menggunakan checksum/sha1 pada mapping).
#   - Menyediakan caching automaton yang mempercepat startup/pipeline pada run selanjutnya.
# Catatan / Perhatian:
#   - Mekanisme ini bergantung pada keberadaan meta['sha1'] yang konsisten dengan cara current_sha dihitung.
#     Jika meta['sha1'] tidak pernah ditulis sebelumnya, automaton akan selalu dibangun ulang.
#   - Pickle automaton bisa besar; simpan di storage yang memadai. Atomic write membantu menghindari file korup.
#   - Jika mapping berubah, pastikan Anda juga memperbarui penulisan meta['sha1'] di langkah yang membuat mapping (biasanya pada tahap penyimpanan mapping).
#   - Dependensi: compute_sha1, atomic_write_pickle, atomic_write_text, json, pickle, datetime, timezone, build_aho_automaton.
def safe_load_or_rebuild_automaton(mapping, automaton_pkl=AUTOMATON_PKL, 
                                   metadata_json=METADATA_JSON):
    # Compute current mapping signature
    current_sha = compute_sha1(json.dumps(sorted(list(mapping.items())), ensure_ascii=False))
    
    # Load metadata
    meta = {}
    if os.path.exists(metadata_json):
        try:
            with open(metadata_json, 'r', encoding='utf-8') as f:
                meta = json.load(f)
        except Exception:
            meta = {}
    
    meta_sha = meta.get('sha1')
    
    # Try to load cached automaton if SHA matches
    if os.path.exists(automaton_pkl) and meta_sha == current_sha:
        try:
            with open(automaton_pkl, 'rb') as f:
                automaton = pickle.load(f)
            print("📂 Loaded cached automaton")
            return automaton
        except Exception:
            print("⚠️ Failed loading automaton cache; rebuilding")
    
    # Rebuild automaton
    automaton = build_aho_automaton(mapping)
    
    # Cache it
    if automaton is not None:
        try:
            atomic_write_pickle(automaton_pkl, automaton)
            meta['automaton_saved_at'] = datetime.now(timezone.utc).isoformat()
            atomic_write_text(metadata_json, json.dumps(meta, indent=2, ensure_ascii=False))
            print("💾 Saved automaton cache")
        except Exception as e:
            print(f"⚠️ Could not cache automaton: {e}")
    
    return automaton

print("✅ Aho-Corasick functions ready!")

✅ Aho-Corasick functions ready!


In [15]:
# Input:
#   - text (str): teks yang mungkin mengandung pengulangan karakter berlebih, mis. "heyyyy" atau "soooo"
# Proses / cara kerja (detail):
#   1. Menggunakan regex re.sub(r'(.)\1{2,}', r'\1\1', text) untuk menggantikan setiap run karakter
#      yang diulang 3 kali atau lebih dengan hanya dua pengulangan.
#      Contoh: "loooool" -> "lool", "yesssss" -> "yess"
#   2. Regex menangkap satu karakter (.) lalu \1{2,} berarti dua atau lebih pengulangan tambahan dari karakter yang sama.
# Output:
#   - (str) teks dengan pengulangan karakter panjang dikurangi menjadi maksimal dua kali berurutan.
# Kegunaan:
#   - Mengurangi variasi token akibat elongation (panjang-perpanjangan) yang sering ditemui di media sosial,
#     sehingga membantu stabilitas tokenisasi dan pencocokan kata/kamus.
#   - Mencegah sparsity pada vocabulary akibat banyak variasi panjang karakter.
def reduce_lengthening(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)
# Input:
#   - text (str): teks yang mungkin mengandung emoji
# Proses / cara kerja (detail):
#   1. Memanggil emoji.demojize(text, delimiters=(" ", " ")) yang menggantikan setiap emoji
#      dengan nama teksnya, contohnya "😊" -> " smiling_face_with_smiling_eyes " (dikelilingi spasi).
#   2. Memilih delimiters spasi agar token nama emoji tidak terikat langsung ke kata lain.
# Output:
#   - (str) representasi teks dengan emoji yang sudah diubah menjadi nama (demojized), atau error jika input tidak sesuai
# Kegunaan:
#   - Membuat emoji menjadi bentuk teks yang dapat diproses lebih mudah oleh pipeline NLP (tokenisasi, pencocokan kata).
#   - Membantu normalisasi teks sehingga emoji dapat dianggap sebagai kata/deskriptor dalam analisis sentimen atau aspek.
def demojize_keep(text):
    return emoji.demojize(text, delimiters=(" ", " "))
# Input:
#   - text (str|None): teks mentah yang akan dibersihkan / dinormalisasi dasar
# Proses / cara kerja (detail):
#   1. Jika text None -> set t = "" else konversi ke str dan lakukan .strip() untuk menghilangkan spasi ujung.
#   2. Gunakan unidecode(t) untuk menghapus aksen/diakritik (mis. "café" -> "cafe").
#   3. Konversi seluruh teks ke huruf kecil (lowercase).
#   4. Hapus URL menggunakan regex re.sub(r'http\S+|www\.\S+', ' ', t) — menggantikan URL dengan spasi.
#   5. Hapus mention dan hashtag dengan re.sub(r'@\w+|#\w+', ' ', t) — menggantikan dengan spasi.
#   6. Panggil reduce_lengthening(t) untuk mereduksi pengulangan karakter berlebih (elongation).
#   7. Panggil demojize_keep(t) untuk mengganti emoji menjadi nama teksnya (dipisahkan spasi).
#   8. Collapse whitespace berlebih menjadi satu spasi dan trim ujung dengan re.sub(r'\s+', ' ', t).strip().
#   9. Kembalikan hasil bersih sebagai string.
# Output:
#   - (str) teks yang sudah dibersihkan secara dasar: tanpa aksen, lowercase, tanpa URL/mention/hashtag,
#     emoji digantikan, elongation dikurangi, whitespace dinormalisasi.
# Kegunaan:
#   - Preprocessing awal sebelum langkah lanjutan (tokenisasi, stopword removal, slang normalization, dsb.)
#   - Mengurangi noise umum dari teks sosial media sehingga pipeline NLP lebih stabil dan hasil analisis lebih konsisten.
# Catatan / Dependensi:
#   - Membutuhkan modul: unidecode.unidecode, emoji.demojize, re.
#   - Jika Anda ingin mempertahankan hashtag/mentions untuk analisis khusus, jangan hapus langkah tersebut.
#   - Untuk kebutuhan multibahasa, pertimbangkan handling khusus (mis. transliteration atau stopword bahasa tertentu).
def basic_clean(text):
    # Handle None or empty
    if text is None or (isinstance(text, str) and not text.strip()):
        return ""
    
    t = str(text).strip()

    # Convert emoji ke text descriptions
    t = demojize_keep(t)
    
    # Unidecode untuk normalize accented characters
    t = unidecode(t)
    
    # Lowercase
    t = t.lower()
    
    # Remove URLs
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    
    # Remove mentions dan hashtags (umumnya tidak relevant untuk sentiment)
    t = re.sub(r'@\w+|#\w+', ' ', t)
    
    # Reduce excessive character repetition
    t = reduce_lengthening(t)
    
    # Normalize whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    
    return t

print("✅ Basic cleaning functions ready!")

✅ Basic cleaning functions ready!


In [16]:
# Token pattern untuk word-level replacement
_token_pat = re.compile(r'\w+', flags=re.UNICODE)
# Input:
#   - text (str): teks yang akan dinormalisasi pada level token
#   - mapping (dict): kamus mapping token (lowercase) -> pengganti (formal)
# Proses / Cara kerja (detail):
#   1. Jika `text` bukan str atau `mapping` falsy, kembalikan `text` apa adanya.
#   2. Fungsi `repl_token` dipakai sebagai callback untuk `_token_pat.sub`:
#        - Untuk tiap token yang dipasangkan oleh regex (kata berbasis \w+), ambil grup token.
#        - Normalize token ke lowercase dan cek apakah ada di `mapping`.
#        - Jika ada, kembalikan replacement dari mapping; jika tidak, kembalikan token asli (preserve case asli untuk non-matching).
#   3. Gunakan `_token_pat.sub(repl_token, text)` untuk mengganti setiap token sesuai mapping.
# Output:
#   - (str) teks hasil setelah penggantian berbasis token.
# Kegunaan:
#   - Cocok untuk mengganti slang atau singkatan yang berdiri sendiri sebagai token (mis. "brb" -> "be right back").
#   - Menghindari penggantian yang mengubah substring di dalam kata (karena hanya mengganti token yang dikenali oleh regex).
def token_based_replace(text, mapping):
    if not isinstance(text, str) or not mapping:
        return text
    
    def repl_token(m):
        tok = m.group(0)
        lower = tok.lower()
        if lower in mapping:
            return mapping[lower]
        return tok
    
    return _token_pat.sub(repl_token, text)
# Input:
#   - text (str): teks sumber yang akan dicari frasa/frasa-slank
#   - automaton (ahocorasick.Automaton): automaton Aho-Corasick yang berisi pola -> (key, replacement)
#   - mapping (dict): kamus mapping (tidak dipakai langsung dalam loop, tetapi disertakan untuk konsistensi API)
#   - whole_word (bool): jika True, hanya cocokkan pola yang berdiri pada batas kata (word-boundary)
# Proses / Cara kerja (detail):
#   1. Jika automaton None atau mapping falsy, kembalikan teks lowercased (jika input str) atau teks asli.
#   2. Konversi teks ke lowercase (`lower_text`) untuk pencocokan case-insensitive.
#   3. Iterasi over `automaton.iter(lower_text)` yang mengembalikan `(end_idx, val)` untuk tiap match.
#        - `val` diasumsikan berisi (key, replacement) sebagaimana saat membangun automaton.
#        - Hitung `start_idx = end_idx - len(key) + 1`.
#        - Jika `whole_word` True, lakukan pemeriksaan boundary kiri/kanan menggunakan regex `\w` supaya
#          pola hanya diterima jika tidak bertemu huruf/angka di sebelahnya.
#        - Jika lolos pemeriksaan (atau whole_word False), tambahkan tuple match (start_idx, end_idx, key, replacement).
#   4. Jika tidak ada match, kembalikan `lower_text`.
#   5. Urutkan `matches` berdasarkan posisi mulai `start` dan preferensi panjang pola (pilih pola lebih panjang dulu ketika overlap).
#   6. Pilih match yang tidak tumpang-tindih secara greedy: iterasi matches dan ambil match bila `start > last_end`.
#   7. Bangun output dengan menggabungkan bagian-bagian teks non-matching dan pengganti (`replacement`) untuk setiap match terpilih.
#   8. Normalisasi whitespace (collapse multiple spaces) dan trim.
# Output:
#   - (str) teks hasil setelah penggantian frasa, dalam lowercase, dengan whitespace dinormalisasi.
# Kegunaan:
#   - Efisien mengganti frasa/slang multi-kata menggunakan Aho-Corasick untuk banyak pola sekaligus.
#   - Lebih cocok untuk frasa yang terdiri dari beberapa token (mis. "gk jelas" -> "tidak jelas") dibanding pencocokan token saja.
# Catatan:
#   - Fungsi mengembalikan teks lowercased; jika ingin mempertahankan kapitalisasi asli, perlu modifikasi tambahan.
#   - Automaton melakukan pencocokan substring; opsi `whole_word` membantu mengurangi false positives pada boundary kata.
def aho_phrase_replace(text, automaton, mapping, whole_word=True):
    if automaton is None or not mapping:
        return text.lower() if isinstance(text, str) else text
    
    lower_text = text.lower()
    matches = []
    
    # Find all matches using automaton
    for end_idx, val in automaton.iter(lower_text):
        key, replacement = val
        start_idx = end_idx - len(key) + 1
        
        if whole_word:
            # Check word boundaries
            before_ok = (start_idx == 0) or (not re.match(r'\w', lower_text[start_idx - 1]))
            after_ok = (end_idx + 1 == len(lower_text)) or (not re.match(r'\w', lower_text[end_idx + 1]))
            
            if before_ok and after_ok:
                matches.append((start_idx, end_idx, key, replacement))
        else:
            matches.append((start_idx, end_idx, key, replacement))
    
    if not matches:
        return lower_text
    
    # Sort matches: earlier position first, then longer match
    matches.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    
    # Resolve overlapping matches (greedy: keep first/longest)
    chosen = []
    last_end = -1
    for s, e, k, v in matches:
        if s > last_end:
            chosen.append((s, e, k, v))
            last_end = e
    
    # Build output dengan replacements
    out_parts = []
    idx = 0
    for s, e, k, v in chosen:
        out_parts.append(lower_text[idx:s])
        out_parts.append(v.lower())
        idx = e + 1
    out_parts.append(lower_text[idx:])
    
    normalized = ''.join(out_parts)
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    
    return normalized
# Input:
#   - text (str): teks yang akan dinormalisasi slangnya
# Proses / Cara kerja (detail):
#   1. Jika `text` bukan str, kembalikan `text` apa adanya.
#   2. Jika global `automaton` dan `SLANG_DICT` tersedia:
#        - Lakukan `token_based_replace` terlebih dahulu pada teks lowercased untuk mengganti token-level slang.
#        - Lanjutkan dengan `aho_phrase_replace` untuk mengganti frasa/slang multi-token (whole_word=True).
#        - Kembalikan hasil akhir (hasil dari tahap frasa).
#   3. Jika automaton atau SLANG_DICT tidak tersedia, fallback sederhana:
#        - Pecah teks dengan `split()` dan ganti tiap token menggunakan SLANG_DICT (jika ada),
#          lalu gabungkan kembali menjadi string.
# Output:
#   - (str) teks yang telah dinormalisasi dari slang (hasilnya dalam lowercase ketika menggunakan automaton-path).
# Kegunaan:
#   - Fungsi utama untuk pipeline normalisasi slang sebelum tahap preprocessing lebih lanjut (tokenisasi, stopword, dsb).
#   - Menggabungkan strategi token-level dan phrase-level untuk akurasi penggantian yang lebih baik.
# Catatan:
#   - Mengandalkan variabel global `automaton` dan `SLANG_DICT`. Pastikan keduanya disiapkan (load mapping -> build automaton).
#   - Jika ingin mempertahankan kapitalisasi/punctuation asli, perlu modifikasi lebih lanjut.
def normalize_slang(text, slang_dict_local=None, automaton_local=None):
    # Use global variables if locals not provided
    mapping = slang_dict_local if slang_dict_local is not None else SLANG_DICT_GLOBAL
    autom = automaton_local if automaton_local is not None else AUTOMATON_GLOBAL
    
    if not isinstance(text, str):
        return text
    
    # Try automaton-based approach (efficient)
    if autom and mapping:
        t1 = token_based_replace(text.lower(), mapping)
        t2 = aho_phrase_replace(t1, autom, mapping, whole_word=True)
        return t2
    
    # Fallback: simple token replacement
    else:
        toks = text.split()
        return " ".join([mapping.get(tok.lower().strip(), tok) for tok in toks])

print("✅ Slang normalization functions ready!")

✅ Slang normalization functions ready!


In [17]:
# Input:
#   - text (str): teks mentah yang akan dipreproses untuk training ABSA (Aspect-Based Sentiment Analysis).
#     Jika bukan string atau kosong/whitespace saja, fungsi akan mengembalikan string kosong.
# Proses / Cara kerja (detail langkah demi langkah):
#   1. Validasi input: bila text bukan str atau hanya whitespace -> return "".
#   2. Panggil basic_clean(text) untuk pembersihan dasar:
#        - transliterasi aksen, lowercase, hapus URL/mention/hashtag, kurangi elongation, demojize, dll.
#   3. Normalisasi slang: panggil normalize_slang(t) yang menggabungkan token-level dan phrase-level replacement
#        menggunakan SLANG_DICT dan automaton Aho-Corasick (jika tersedia).
#   4. Tokenisasi sederhana: split() pada spasi untuk menghasilkan list token `toks`.
#   5. Hapus stopword & token pendek:
#        - Filter token yang ada di INDO_STOPWORDS dan token berdimensi 1 karakter (len(tok) > 1).
#   6. Stemming: terapkan stemmer.stem(tok) pada setiap token tersisa untuk mengurangi kata ke bentuk dasar.
#   7. Gabungkan kembali token menjadi string terpisah spasi dan kembalikan hasil.
# Output:
#   - (str) teks yang sudah dipreproses: dibersihkan, slang dinormalisasi, stopword dihapus, dan telah distem.
#     Jika input tidak valid, mengembalikan "".
# Kegunaan:
#   - Pipeline preprocessing yang lebih agresif untuk persiapan data training ABSA (Phase 3).
#   - Menghasilkan fitur teks yang lebih ringkas dan konsisten untuk pelatihan model (mis. TF-IDF, embedding, atau model ML/DL).
# Catatan / Dependensi:
#   - Mengandalkan fungsi/variabel eksternal: basic_clean, normalize_slang, INDO_STOPWORDS (set/list), stemmer.
#   - Pastikan `stemmer` memiliki metode `.stem()` (mis. nltk.SnowballStemmer atau Sastrawi stemmer).
#   - Jika ingin mempertahankan konteks kapitalisasi atau emotikon sebagai fitur, sesuaikan langkah-langkah di atas.
def preprocess_classic(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    t = basic_clean(text)
    t = normalize_slang(t)
    toks = t.split()
    toks = [tok for tok in toks if tok not in INDO_STOPWORDS_FINAL and len(tok) > 1]
    toks = [stemmer.stem(tok) for tok in toks]
    return ' '.join(toks)

print("✅ Complete preprocessing pipeline ready!")
print("\n" + "="*70)
print("🔧 LOADING RESOURCES")
print("="*70)

print("\nThis will:")
print("  1. Fetch slang dictionaries from GitHub (or use cache)")
print("  2. Build Aho-Corasick automaton for efficient matching")
print("  3. Set up global variables for preprocessing")
print("\nPlease wait...")

# Load slang dictionary (dengan caching)
SLANG_DICT_GLOBAL, meta_info = load_mapping_with_metadata(
    SLANG_URLS,
    cache_csv=LOCAL_SLANG_CSV,
    cache_pkl=LOCAL_SLANG_PKL,
    metadata_json=METADATA_JSON,
    session=make_session_with_retry(),
    max_age_days=30
)

# Build atau load automaton
AUTOMATON_GLOBAL = safe_load_or_rebuild_automaton(
    SLANG_DICT_GLOBAL,
    automaton_pkl=AUTOMATON_PKL,
    metadata_json=METADATA_JSON
)

print("\n" + "="*70)
print("✅ RESOURCES LOADED SUCCESSFULLY!")
print("="*70)
print(f"\n📊 Statistics:")
print(f"  • Slang dictionary entries: {len(SLANG_DICT_GLOBAL):,}")
print(f"  • Automaton ready: {AUTOMATON_GLOBAL is not None}")
print(f"  • Stopwords (after filtering): {len(INDO_STOPWORDS_FINAL):,}")
print("\n🎯 Ready untuk preprocessing!")

✅ Complete preprocessing pipeline ready!

🔧 LOADING RESOURCES

This will:
  1. Fetch slang dictionaries from GitHub (or use cache)
  2. Build Aho-Corasick automaton for efficient matching
  3. Set up global variables for preprocessing

Please wait...
📥 Fetching slang dictionaries from online sources...
  ✅ colloquial-indonesian-lexicon.csv: fetched 4330 entries
  ✅ new_kamusalay.csv: fetched 15166 entries
💾 Saved slang dictionary: 15623 entries
⚙️  Built Aho-Corasick automaton: 15623 patterns in 0.005s
💾 Saved automaton cache

✅ RESOURCES LOADED SUCCESSFULLY!

📊 Statistics:
  • Slang dictionary entries: 15,623
  • Automaton ready: True
  • Stopwords (after filtering): 124

🎯 Ready untuk preprocessing!


In [18]:
df['content'] = df['content'].apply(preprocess_classic)
print("\n" + "="*70)
print("💾 SAVING PREPROCESSED DATASET")
print("="*70)

output_path = '/kaggle/working/phase2_annotated_preprocessed_final.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
df.head()


💾 SAVING PREPROCESSED DATASET


,reviewId,userName,userImage,content,score,reviewCreatedVersion,at,replyContent,repliedAt,matched_aspects,Aplikasi,Pengiriman,Produk,Harga,Pembayaran,Layanan Pelanggan (CS),Penjual,text_length
0,31ad33fd-6d00-45d6-b06d-7f69a956c7c0,Aliva Olina,https://play-lh.googleusercontent.com/a/ACg8oc...,barang2x bagus datang tepat waktu cuma ongkos ...,5,3.59.43,2025-10-21 9:38:00,Hi kak Aliva Olina maaf ya terkait keluhan kam...,2025-10-21 10:47:55,"Pengiriman,Produk,Harga",0,P,P,N,0,0,0,70
1,85ed5a52-eabe-4f20-b849-ee7d9e184277,arum galih,https://play-lh.googleusercontent.com/a/ACg8oc...,kirim lama pol,1,,2025-10-21 9:37:38,Hai kak arum galih maaf ya bikin ga nyaman ter...,2025-10-21 10:29:48,Pengiriman,0,N,0,0,0,0,0,20
2,b2052463-4a1b-4315-bfa2-4fd139b7136b,Sandi Rama,https://play-lh.googleusercontent.com/a/ACg8oc...,bagus aplikasi manfaat sulit cari barang langsung,5,3.60.24,2025-10-21 9:36:32,"Hi Kak Sandi Rama, makasih bgt ya review dan b...",2025-10-21 10:07:54,Aplikasi,P,0,0,0,0,0,0,74
3,61843fd3-cfe2-4fa2-8199-b3f27db98e7c,Nar Tiwen,https://play-lh.googleusercontent.com/a/ACg8oc...,padahal cari lokasi sama toko indramayu pesan ...,2,,2025-10-21 9:35:22,Hai kak Nar Tiwen\nmaaf yaa udh buat ga nyaman...,2025-10-21 10:11:01,Pengiriman,0,N,0,0,0,0,0,236
4,c760828f-b897-4dfb-aa52-6225d2cfe7c2,Darpin Er,https://play-lh.googleusercontent.com/a/ACg8oc...,kasih bintang lima cara masuk mudah tidak bany...,5,,2025-10-21 9:35:01,"Hi Kak Darpin Er , makasih ya buat bintang nya...",2025-10-21 10:10:56,"Aplikasi,Produk",P,0,P,0,0,0,0,257


In [19]:
# ============================================================================
# Quantify Code-Mixing Problem: Identify Javanese/Regional Language Words
# ============================================================================

print("="*70)
print("🔍 ANALYZING POTENTIAL JAVANESE/REGIONAL LANGUAGE WORDS")
print("="*70)

# Step 1: Build Indonesian vocabulary baseline
print("\n📚 Building Indonesian vocabulary baseline...")

indonesian_vocab = set(INDO_STOPWORDS_FINAL)  # Start with stopwords

# Add common Indonesian words from preprocessed data (sudah cleaned)
all_tokens = []
for text in df['content']:
    if isinstance(text, str):
        all_tokens.extend(text.split())

# Build frequency distribution
from collections import Counter
word_freq_distribution = Counter(all_tokens)

# Add top 1000 most common words to Indonesian vocab
# (Assuming these are legitimate Indonesian words karena preprocessing sudah clean)
common_indo_words = set([word for word, freq in word_freq_distribution.most_common(1000)])
indonesian_vocab.update(common_indo_words)

print(f"  • Indonesian vocabulary size: {len(indonesian_vocab):,} words")
print(f"  • Total tokens analyzed: {len(all_tokens):,}")
print(f"  • Unique tokens: {len(set(all_tokens)):,}")

# Step 2: Identify OOV words from ORIGINAL content (before preprocessing)
print("\n🔍 Identifying OOV words in original reviews...")

oov_words = []
for text in df['content']:
    if not isinstance(text, str):
        continue
    # Basic cleaning untuk consistency
    text_clean = text.lower()
    tokens = text_clean.split()
    for token in tokens:
        # Remove punctuation
        token_clean = re.sub(r'[^\w]', '', token)
        if len(token_clean) > 2 and token_clean not in indonesian_vocab:
            oov_words.append(token_clean)

# Frequency analysis
oov_freq = Counter(oov_words)

print(f"\n📊 Found {len(oov_freq):,} unique OOV words")
print(f"📊 Total OOV occurrences: {len(oov_words):,}")

# Step 3: Display top OOV words
print("\n" + "="*70)
print("TOP 50 OOV WORDS (Potential Regional Language / Typos / Slang)")
print("="*70)

for i, (word, count) in enumerate(oov_freq.most_common(50), 1):
    percentage = (count / len(df)) * 100
    print(f"  {i:2d}. {word:20s} : {count:4d} times ({percentage:.2f}% of reviews)")

# Step 4: Calculate impact metrics
print("\n" + "="*70)
print("📊 CODE-MIXING IMPACT METRICS")
print("="*70)

# Reviews containing OOV words
reviews_with_oov = 0
for text in df['content']:
    if not isinstance(text, str):
        continue
    text_clean = text.lower()
    tokens = [re.sub(r'[^\w]', '', t) for t in text_clean.split()]
    if any(token in oov_freq for token in tokens if len(token) > 2):
        reviews_with_oov += 1

oov_percentage = (reviews_with_oov / len(df)) * 100

print(f"\n  • Reviews with OOV words: {reviews_with_oov} ({oov_percentage:.1f}%)")
print(f"  • Average OOV per review: {len(oov_words) / len(df):.2f}")
print(f"  • OOV ratio in corpus: {len(oov_words) / len(all_tokens) * 100:.2f}%")

# Step 5: Manual review recommendations
print("\n" + "="*70)
print("💡 NEXT STEPS FOR MANUAL REVIEW")
print("="*70)
print("\nReview the Top 50 OOV words above and categorize them:")
print("  1. ✅ Javanese words → Add to normalization dictionary")
print("  2. ✅ Typos/variations → Add to slang dictionary")
print("  3. ✅ Domain terms → Keep as-is (valid Indonesian)")
print("  4. ❌ Noise/junk → Can be ignored")
print("\nFocus on words dengan frequency > 5 untuk maximum impact.")

# Optional: Export untuk manual review
oov_df = pd.DataFrame(oov_freq.most_common(100), columns=['word', 'frequency'])
oov_df['percentage'] = (oov_df['frequency'] / len(df)) * 100
oov_df.to_csv('/kaggle/working/oov_analysis.csv', index=False, encoding='utf-8-sig')
print(f"\n💾 Saved top 100 OOV words to: /kaggle/working/oov_analysis.csv")

🔍 ANALYZING POTENTIAL JAVANESE/REGIONAL LANGUAGE WORDS

📚 Building Indonesian vocabulary baseline...
  • Indonesian vocabulary size: 1,079 words
  • Total tokens analyzed: 14,072
  • Unique tokens: 2,001

🔍 Identifying OOV words in original reviews...

📊 Found 951 unique OOV words
📊 Total OOV occurrences: 977

TOP 50 OOV WORDS (Potential Regional Language / Typos / Slang)
   1. hearteyes            :   16 times (1.55% of reviews)
   2. mediumlight          :    7 times (0.68% of reviews)
   3. ecommerce            :    2 times (0.19% of reviews)
   4. starstruck           :    2 times (0.19% of reviews)
   5. upsidedown           :    2 times (0.19% of reviews)
   6. zippermouth          :    2 times (0.19% of reviews)
   7. spxnya               :    2 times (0.19% of reviews)
   8. two                  :    1 times (0.10% of reviews)
   9. batu                 :    1 times (0.10% of reviews)
  10. bamyak               :    1 times (0.10% of reviews)
  11. kunjung              :    1 t

In [20]:
import pandas as pd
import textwrap

def debug_preprocess_single(text,
                            slang_mapping=None,
                            automaton=None):
    """
    Debug preprocessing sesuai pipeline eksplisit:

    ORIGINAL
     ├─ BASIC CLEAN
     ├─ TOKEN-LEVEL SLANG (from ORIGINAL)
     ├─ PHRASE-LEVEL SLANG (from ORIGINAL)
     ├─ CLEAN + SLANG NORMALIZATION
     ├─ TOKENIZATION
     ├─ STOPWORD REMOVAL (before / after)
     ├─ STEMMING (before / after)
     └─ FINAL PREPROCESS
    """

    # --- fallback global ---
    mapping = slang_mapping if slang_mapping is not None else globals().get("SLANG_DICT_GLOBAL")
    autom = automaton if automaton is not None else globals().get("AUTOMATON_GLOBAL")

    basic = globals().get("basic_clean")
    token_replace_fn = globals().get("token_based_replace")
    phrase_replace_fn = globals().get("aho_phrase_replace")
    norm_slang = globals().get("normalize_slang")
    stemmer_obj = globals().get("stemmer")
    stopwords_set = globals().get("INDO_STOPWORDS_FINAL", set())

    result = {}

    # 0) ORIGINAL
    result["original"] = text

    # 1) BASIC CLEAN (ONLY CLEANING)
    try:
        basic_cleaned = basic(text)
    except Exception:
        basic_cleaned = text.lower()
    result["basic_cleaned"] = basic_cleaned

    # 2) TOKEN-LEVEL SLANG REPLACE (FROM ORIGINAL)
    try:
        token_level = token_replace_fn(text.lower(), mapping)
    except Exception:
        token_level = text.lower()
    result["token_level_replaced"] = token_level

    # 3) PHRASE-LEVEL SLANG REPLACE (FROM ORIGINAL)
    try:
        phrase_level = phrase_replace_fn(
            text.lower(), autom, mapping, whole_word=True
        )
    except Exception:
        phrase_level = text.lower()
    result["phrase_level_replaced"] = phrase_level

    # 4) CLEAN + SLANG NORMALIZATION (PIPELINE FINAL)
    try:
        clean_plus_slang = norm_slang(
            basic_cleaned,
            slang_dict_local=mapping,
            automaton_local=autom
        )
    except Exception:
        clean_plus_slang = basic_cleaned
    result["clean_plus_slang"] = clean_plus_slang

    # 5) TOKENIZATION
    tokens_all = clean_plus_slang.split()
    result["tokens_before_stopword"] = tokens_all

    # 6) STOPWORD REMOVAL
    tokens_after_stopword = [
        t for t in tokens_all
        if t not in stopwords_set and len(t) > 1
    ]
    result["tokens_after_stopword"] = tokens_after_stopword
    result["text_after_stopword"] = " ".join(tokens_after_stopword)

    # 7) STEMMING
    if stemmer_obj:
        tokens_after_stemming = [stemmer_obj.stem(t) for t in tokens_after_stopword]
    else:
        tokens_after_stemming = tokens_after_stopword

    result["tokens_after_stemming"] = tokens_after_stemming
    result["text_after_stemming"] = " ".join(tokens_after_stemming)


    # 8) FINAL PREPROCESS STRING
    try:
        if "preprocess_classic" in globals():
            final = globals()["preprocess_classic"](text)
        else:
            final = " ".join(stemmed_tokens)
    except Exception:
        final = " ".join(stemmed_tokens)

    result["final_preprocessed"] = final

    return result


def pretty_print_debug(d, width=80):
    sep = "-" * width
    print(sep)

    print("ORIGINAL:")
    print(textwrap.fill(d["original"], width))
    print()

    print("1) BASIC CLEAN:")
    print(textwrap.fill(d["basic_cleaned"], width))
    print()

    print("2) TOKEN-LEVEL SLANG (from ORIGINAL):")
    print(textwrap.fill(d["token_level_replaced"], width))
    print()

    print("3) PHRASE-LEVEL SLANG (from ORIGINAL):")
    print(textwrap.fill(d["phrase_level_replaced"], width))
    print()

    print("4) CLEAN + SLANG NORMALIZATION:")
    print(textwrap.fill(d["clean_plus_slang"], width))
    print()

    print("5) TOKENS BEFORE STOPWORD REMOVAL:")
    print(d["tokens_before_stopword"])
    print()

    print("6) TOKENS AFTER STOPWORD REMOVAL:")
    print(d["tokens_after_stopword"])
    print()

    print("   TEXT AFTER STOPWORD REMOVAL:")
    print(textwrap.fill(d["text_after_stopword"], width))
    print()

    print("7) TOKENS AFTER STEMMING:")
    print(d["tokens_after_stemming"])
    print()

    print("   TEXT AFTER STEMMING:")
    print(textwrap.fill(d["text_after_stemming"], width))
    print()


    print("8) FINAL PREPROCESSED STRING:")
    print(d["final_preprocessed"])

    print(sep + "\n")


def debug_preprocess_batch(texts):
    rows = []
    for t in texts:
        snap = debug_preprocess_single(t)
        rows.append({
            "original": snap["original"],
            "basic_cleaned": snap["basic_cleaned"],
            "token_level": snap["token_level_replaced"],
            "phrase_level": snap["phrase_level_replaced"],
            "clean_plus_slang": snap["clean_plus_slang"],
            "tokens_after_stopword": " ".join(snap["tokens_after_stopword"]),
            "tokens_after_stemming": " ".join(snap["tokens_after_stemming"]),
            "final_preprocessed": snap["final_preprocessed"]
        })
    return pd.DataFrame(rows)


# =============================
# CONTOH PENGGUNAAN
# =============================

example_reviews = [
    "Gokil bgt makanannya! enakkkk banget, tapi pelayanannya gk jelas :( 😭",
    "Produk OK, tp kualitasnya kurang memuaskan, blm sesuai ekspektasi.",
    "Saya suka, recommended! harga murah, pelayanan ramah :) #happy",
    "nggak recomended. barang datang rusak dan customer service ga bantu sama sekali!",
    "wah mantul bener, puassssss!!! 😍🔥"
]

for r in example_reviews:
    pretty_print_debug(debug_preprocess_single(r))

df_debug = debug_preprocess_batch(example_reviews)
print(df_debug.to_markdown(index=False))


--------------------------------------------------------------------------------
ORIGINAL:
Gokil bgt makanannya! enakkkk banget, tapi pelayanannya gk jelas :( 😭

1) BASIC CLEAN:
gokil bgt makanannya! enakk banget, tapi pelayanannya gk jelas :(
loudly_crying_face

2) TOKEN-LEVEL SLANG (from ORIGINAL):
gila banget makanannya! enakkkk banget, tapi pelayanannya tidak jelas :( 😭

3) PHRASE-LEVEL SLANG (from ORIGINAL):
gila banget makanannya! enakkkk banget, tapi pelayanannya tidak jelas :( 😭

4) CLEAN + SLANG NORMALIZATION:
gila banget makanannya! enakk banget, tapi pelayanannya tidak jelas :(
loudly_crying_face

5) TOKENS BEFORE STOPWORD REMOVAL:
['gila', 'banget', 'makanannya!', 'enakk', 'banget,', 'tapi', 'pelayanannya', 'tidak', 'jelas', ':(', 'loudly_crying_face']

6) TOKENS AFTER STOPWORD REMOVAL:
['gila', 'makanannya!', 'enakk', 'banget,', 'pelayanannya', 'tidak', 'jelas', ':(', 'loudly_crying_face']

   TEXT AFTER STOPWORD REMOVAL:
gila makanannya! enakk banget, pelayanannya tidak j